# Flow Data Completeness Check

This notebook validates completeness of annotations in `data/ipnall_flo.json` and checks that the referenced flow data exists under `data/flow`.

In [5]:
import json
from pathlib import Path

# Annotation index (from dataset root)
ANNOTATION_PATH = Path("../data/raw/ipn_hand/ipnall_flo.json")
# Flow data directory (must exist for each referenced sequence)
FLOW_DIR = Path("../data/raw/ipn_hand/flow")

REQUIRED_TOP_KEYS = {"labels", "database"}
REQUIRED_ANNOTATION_KEYS = {"label", "start_frame", "end_frame"}
VALID_SUBSETS = {"training", "validation"}

In [6]:
# Load annotations
with open(ANNOTATION_PATH) as f:
    data = json.load(f)

print("Annotations:", ANNOTATION_PATH)
print("Flow data:  ", FLOW_DIR)
print("Top-level keys:", list(data.keys()))
print("Labels:", data["labels"])
print("Database entries:", len(data["database"]))

Annotations: ../data/raw/ipn_hand/ipnall_flo.json
Flow data:   ../data/raw/ipn_hand/flow
Top-level keys: ['labels', 'database']
Labels: ['D0X', 'B0A', 'B0B', 'G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11']
Database entries: 5519


## 1. Top-level structure

In [7]:
top_keys = set(data.keys())
missing_top = REQUIRED_TOP_KEYS - top_keys
extra_top = top_keys - REQUIRED_TOP_KEYS

if not missing_top and not extra_top:
    print("✓ Top-level keys: all required keys present, no extra keys")
else:
    if missing_top:
        print("✗ Missing top-level keys:", missing_top)
    if extra_top:
        print("⚠ Extra top-level keys:", extra_top)

✓ Top-level keys: all required keys present, no extra keys


## 2. Database entry structure (subset + annotations)

In [8]:
missing_subset = []
missing_annotations = []
invalid_subset = []
missing_ann_keys = []
invalid_labels = []
invalid_frame_ranges = []
labels_set = set(data["labels"])

for key, entry in data["database"].items():
    if "subset" not in entry:
        missing_subset.append(key)
    elif entry["subset"] not in VALID_SUBSETS:
        invalid_subset.append((key, entry["subset"]))
    if "annotations" not in entry:
        missing_annotations.append(key)
    else:
        ann = entry["annotations"]
        for req in REQUIRED_ANNOTATION_KEYS:
            if req not in ann:
                missing_ann_keys.append((key, req))
                break
        else:
            if ann["label"] not in labels_set:
                invalid_labels.append((key, ann["label"]))
            try:
                start, end = int(ann["start_frame"]), int(ann["end_frame"])
                if start > end:
                    invalid_frame_ranges.append((key, start, end))
            except (ValueError, TypeError):
                invalid_frame_ranges.append((key, ann["start_frame"], ann["end_frame"]))

def report(name, issues, limit=5):
    n = len(issues)
    if n == 0:
        print(f"✓ {name}: no issues")
    else:
        print(f"✗ {name}: {n} issue(s)")
        for x in issues[:limit]:
            print(f"    {x}")
        if n > limit:
            print(f"    ... and {n - limit} more")

report("Missing 'subset'", missing_subset)
report("Invalid subset value", invalid_subset)
report("Missing 'annotations'", missing_annotations)
report("Missing annotation keys", missing_ann_keys)
report("Label not in labels list", invalid_labels)
report("Invalid frame range (start > end or non-numeric)", invalid_frame_ranges)

✓ Missing 'subset': no issues
✓ Invalid subset value: no issues
✓ Missing 'annotations': no issues
✓ Missing annotation keys: no issues
✓ Label not in labels list: no issues
✓ Invalid frame range (start > end or non-numeric): no issues


## 3. Cross-check: flow data under data/flow

Each database key (e.g. `./flow/1CM1_4_R_#229^2`) refers to a segment; the actual flow files live in `data/flow/<sequence_id>/`. We check that every referenced sequence has a directory under `data/flow`.

In [9]:
def key_to_flow_dir(key: str) -> Path:
    """Map DB key (e.g. ./flow/1CM1_4_R_#229^2) to data/flow/<sequence_id>."""
    rest = key.removeprefix("./flow/").removeprefix("flow/")
    base = rest.split("^")[0] if "^" in rest else rest
    return FLOW_DIR / base

missing_flow_dirs = []
for key in data["database"]:
    flow_path = key_to_flow_dir(key)
    if not flow_path.is_dir():
        missing_flow_dirs.append((key, str(flow_path)))

# Unique missing base dirs (multiple segments can share one dir)
missing_dirs_unique = sorted(set(p for _, p in missing_flow_dirs))
entries_missing_flow = len(missing_flow_dirs)

if not missing_flow_dirs:
    print("✓ All entries have flow data under data/flow")
else:
    print(f"✗ {entries_missing_flow} entry/entries reference missing flow dir(s):")
    for key, path in missing_flow_dirs[:10]:
        print(f"    {key} -> {path}")
    if entries_missing_flow > 10:
        print(f"    ... and {entries_missing_flow - 10} more")
    print(f"\nUnique missing dirs: {len(missing_dirs_unique)}")

✓ All entries have flow data under data/flow


## 4. Summary counts and subset distribution

In [10]:
from collections import Counter

subset_counts = Counter(e["subset"] for e in data["database"].values())
label_counts = Counter(e["annotations"]["label"] for e in data["database"].values())

print("Subset distribution:")
for s, c in subset_counts.most_common():
    print(f"  {s}: {c}")
print("\nLabel distribution:")
for lbl, c in label_counts.most_common():
    print(f"  {lbl}: {c}")

Subset distribution:
  training: 4039
  validation: 1480

Label distribution:
  D0X: 1305
  B0A: 1009
  B0B: 1004
  G04: 201
  G11: 200
  G05: 200
  G03: 200
  G02: 200
  G08: 200
  G06: 200
  G10: 200
  G09: 200
  G07: 200
  G01: 200


## 5. Overall completeness

In [12]:
keys_with_issues = set()
for lst in (missing_subset, missing_annotations, invalid_subset, invalid_labels, invalid_frame_ranges):
    for x in lst:
        keys_with_issues.add(x[0] if isinstance(x, tuple) else x)
for key, _ in missing_ann_keys:
    keys_with_issues.add(key)
for key, _ in missing_flow_dirs:
    keys_with_issues.add(key)

total_entries = len(data["database"])
entries_with_issues = len(keys_with_issues)
complete = total_entries - entries_with_issues

print(f"Total database entries: {total_entries}")
print(f"Entries with no issues: {complete}")
print(f"Entries with issues:    {entries_with_issues}")
if total_entries:
    pct = 100 * complete / total_entries
    print(f"Completeness:           {pct:.1f}%")
    print("\n" + ("✓ Data is complete." if entries_with_issues == 0 else "✗ Some entries have completeness issues (see above)."))

Total database entries: 5519
Entries with no issues: 5519
Entries with issues:    0
Completeness:           100.0%

✓ Data is complete.
